# ML-05 — Feature Vector and Leakage/Privacy Check

## 1. Build the Feature Vector

All features are engineered strictly prior to the decision cutoff (`report_date <= 2026-03-21`) at the `(client_hash_id, content_hash_id)` grain on `month=2026-03` (`gsc_data_available IS TRUE`), predicting whether visible URLs suffer a $\ge 20\%$ drop in mean daily impressions during the forward outcome window (`2026-03-22` through `2026-03-31`, `is_future_decay`).

In [2]:
import os
import getpass
import duckdb
import numpy as np
import pandas as pd

# 1. DuckDB & Hugging Face Authentication (aligned with w03_data_contract.ipynb)
HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = getpass.getpass("Cole seu Hugging Face READ token (HF_TOKEN): ")

assert HF_TOKEN and len(HF_TOKEN.strip()) > 10, "HF_TOKEN inválido ou ausente."
os.environ["HF_TOKEN"] = HF_TOKEN.strip()

con = duckdb.connect(database=":memory:")
con.execute(f"CREATE OR REPLACE SECRET hf_secret (TYPE huggingface, TOKEN '{HF_TOKEN.strip()}');")

MONTH = "2026-03"
DATASET_PATH = f"hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={MONTH}/*.parquet"

# 2. Causal Feature Engineering Query (strictly <= 2026-03-21; outcome > 2026-03-21)
query_feature_vector = f"""
WITH panel AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        NULLIF(gsc_avg_position, 0) AS clean_position,
        CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE NULL END AS valid_ga4_sessions,
        CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE NULL END AS valid_ga4_engaged,
        ga4_data_available
    FROM read_parquet('{DATASET_PATH}')
    WHERE gsc_data_available IS TRUE
),
agg AS (
    SELECT
        client_hash_id,
        content_hash_id,

        -- Pre-decision observation window (2026-03-01 to 2026-03-21)
        COUNT(*) FILTER (WHERE report_date <= DATE '2026-03-21') AS raw_active_days_pre,
        AVG(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-21') AS raw_impressions_21d_avg,
        STDDEV_SAMP(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-21') AS raw_impressions_21d_std,
        SUM(gsc_clicks) FILTER (WHERE report_date <= DATE '2026-03-21') AS raw_clicks_21d_sum,
        AVG(clean_position) FILTER (WHERE report_date <= DATE '2026-03-21') AS raw_position_21d_avg,
        MIN(clean_position) FILTER (WHERE report_date <= DATE '2026-03-21') AS raw_best_position_21d,

        -- Trailing 7d window right before cutoff (2026-03-15 to 2026-03-21) vs early window (2026-03-01 to 2026-03-14)
        AVG(gsc_impressions) FILTER (WHERE report_date BETWEEN DATE '2026-03-15' AND DATE '2026-03-21') AS raw_impressions_last7d_avg,
        AVG(gsc_impressions) FILTER (WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14') AS raw_impressions_early14d_avg,

        -- Pre-decision CTR (%)
        100.0 * SUM(gsc_clicks) FILTER (WHERE report_date <= DATE '2026-03-21')
            / NULLIF(SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-21'), 0) AS raw_ctr_21d_pct,

        -- GA4 telemetry strictly filtered by ga4_data_available IS TRUE
        AVG(valid_ga4_sessions) FILTER (WHERE report_date <= DATE '2026-03-21') AS raw_ga4_sessions_21d_avg,
        100.0 * SUM(valid_ga4_engaged) FILTER (WHERE report_date <= DATE '2026-03-21')
            / NULLIF(SUM(valid_ga4_sessions) FILTER (WHERE report_date <= DATE '2026-03-21'), 0) AS raw_ga4_engagement_pct,
        MAX(CASE WHEN report_date <= DATE '2026-03-21' AND ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS raw_is_ga4_connected,

        -- Forward outcome window (2026-03-22 to 2026-03-31) — NEVER used as a feature
        AVG(gsc_impressions) FILTER (WHERE report_date > DATE '2026-03-21') AS post_impressions_mean
    FROM panel
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    *,
    CASE
        WHEN post_impressions_mean <= 0.80 * raw_impressions_21d_avg THEN 1
        ELSE 0
    END AS is_future_decay
FROM agg
WHERE raw_impressions_21d_avg >= 10.0
  AND post_impressions_mean IS NOT NULL;
"""

df_raw = con.execute(query_feature_vector).df()

# 3. Deterministic Missing Value Handling & Feature Transformations
df_feat = df_raw.copy()

# Explicit missingness & coverage flags prior to imputation
df_feat["has_full_21d_history"] = (df_feat["raw_active_days_pre"] >= 21).astype(int)
df_feat["is_position_missing"] = df_feat["raw_position_21d_avg"].isna().astype(int)
df_feat["is_ga4_connected"] = df_feat["raw_is_ga4_connected"].fillna(0).astype(int)

# Numeric fills (neutral/domain-consistent)
pos_median = df_feat["raw_position_21d_avg"].median()
df_feat["impressions_21d_avg"] = df_feat["raw_impressions_21d_avg"].fillna(0.0)
df_feat["impressions_21d_cv"] = (
    (df_feat["raw_impressions_21d_std"].fillna(0.0) / df_feat["impressions_21d_avg"].replace(0, np.nan))
    .fillna(0.0)
    .clip(0.0, 5.0)
)
df_feat["clicks_21d_sum"] = df_feat["raw_clicks_21d_sum"].fillna(0.0)
df_feat["position_21d_avg"] = df_feat["raw_position_21d_avg"].fillna(pos_median)
df_feat["best_position_21d"] = df_feat["raw_best_position_21d"].fillna(pos_median)
df_feat["ctr_21d_pct"] = df_feat["raw_ctr_21d_pct"].fillna(0.0)
df_feat["ga4_sessions_21d_avg"] = df_feat["raw_ga4_sessions_21d_avg"].fillna(0.0)
df_feat["ga4_engagement_pct"] = df_feat["raw_ga4_engagement_pct"].fillna(0.0).clip(0.0, 100.0)

# Momentum ratio: trailing 7d vs early 14d impressions (1.0 = neutral on cold-start)
denom_early = df_feat["raw_impressions_early14d_avg"].replace(0, np.nan)
df_feat["impressions_momentum_ratio"] = (
    (df_feat["raw_impressions_last7d_avg"] / denom_early)
    .replace([np.inf, -np.inf], np.nan)
    .fillna(1.0)
    .clip(0.0, 10.0)
)

# 4. Categorical Engineering & Explicit One-Hot Encoding
def bucket_serp_tier(pos: float, is_missing: int) -> str:
    if is_missing == 1:
        return "unranked"
    if pos <= 3.0:
        return "top_3"
    if pos <= 10.0:
        return "page_1_rest"
    if pos <= 20.0:
        return "striking_distance"
    return "page_3_plus"

df_feat["serp_tier_pre"] = [
    bucket_serp_tier(p, m)
    for p, m in zip(df_feat["position_21d_avg"], df_feat["is_position_missing"])
]

tier_categories = ["top_3", "page_1_rest", "striking_distance", "page_3_plus", "unranked"]
df_feat["serp_tier_pre"] = pd.Categorical(df_feat["serp_tier_pre"], categories=tier_categories)
dummies_tier = pd.get_dummies(df_feat["serp_tier_pre"], prefix="tier", dtype=int)

numeric_feature_cols = [
    "impressions_21d_avg",
    "impressions_21d_cv",
    "clicks_21d_sum",
    "position_21d_avg",
    "best_position_21d",
    "ctr_21d_pct",
    "impressions_momentum_ratio",
    "ga4_sessions_21d_avg",
    "ga4_engagement_pct",
    "is_ga4_connected",
    "has_full_21d_history",
    "is_position_missing",
]

X = pd.concat([df_feat[numeric_feature_cols], dummies_tier], axis=1)
y = df_feat["is_future_decay"].astype(int)
groups_client = df_feat["client_hash_id"]

FEATURE_COLS = list(X.columns)

print(f"Observed eligible URLs (impressions_21d_avg >= 10): {len(X):,}")
print(f"Distinct clients represented: {groups_client.nunique()}")
print(f"Positive decay rate (is_future_decay == 1): {y.mean():.3f}")
print(f"Total engineered features in X: {len(FEATURE_COLS)}")
print("Feature columns:", FEATURE_COLS)

Observed eligible URLs (impressions_21d_avg >= 10): 74,962
Distinct clients represented: 39
Positive decay rate (is_future_decay == 1): 0.316
Total engineered features in X: 17
Feature columns: ['impressions_21d_avg', 'impressions_21d_cv', 'clicks_21d_sum', 'position_21d_avg', 'best_position_21d', 'ctr_21d_pct', 'impressions_momentum_ratio', 'ga4_sessions_21d_avg', 'ga4_engagement_pct', 'is_ga4_connected', 'has_full_21d_history', 'is_position_missing', 'tier_top_3', 'tier_page_1_rest', 'tier_striking_distance', 'tier_page_3_plus', 'tier_unranked']


## 2. Feature Notes (Meaning, Missingness, Categorical, Available-When?)

All features are strictly bounded by the `2026-03-21` cutoff (`report_date <= DATE '2026-03-21'`) and computed only over rows with `gsc_data_available IS TRUE` (and `ga4_data_available IS TRUE` for GA4 telemetry).

| Feature / Group | Type & Encoding | Meaning | Missing Value Handling | Available-When? |
| :--- | :--- | :--- | :--- | :--- |
| `impressions_21d_avg` | Float (continuous) | Mean daily GSC impressions per URL in `[Mar 1, Mar 21]`. | Filtered `>= 10.0`; `fillna(0.0)` safeguard. | Strictly prior: `<= 2026-03-21`. |
| `impressions_21d_cv` | Float (`[0, 5]`) | Coefficient of variation (`std / mean`) of daily impressions. | `0.0` when single-day observation (`std` is `NULL`). | Strictly prior: `<= 2026-03-21`. |
| `clicks_21d_sum` | Float (continuous) | Cumulative organic clicks per URL in `[Mar 1, Mar 21]`. | Filled with `0.0`. | Strictly prior: `<= 2026-03-21`. |
| `position_21d_avg` | Float (continuous) | Mean SERP rank (`NULLIF(gsc_avg_position, 0)`) in `[Mar 1, Mar 21]`. | Imputed with training/panel median + `is_position_missing=1`. | Strictly prior: `<= 2026-03-21`. |
| `best_position_21d` | Float (continuous) | Best (minimum) daily SERP rank observed in `[Mar 1, Mar 21]`. | Imputed with panel median when unranked. | Strictly prior: `<= 2026-03-21`. |
| `ctr_21d_pct` | Float (`[0, 100]`) | Pre-cutoff CTR (`100 * sum_clicks / sum_impressions`, ×100 scale). | `NULLIF(..., 0)` prevents zero division; filled with `0.0`. | Strictly prior: `<= 2026-03-21`. |
| `impressions_momentum_ratio` | Float (`[0, 10]`) | Trailing 7d (`Mar 15–21`) vs. early 14d (`Mar 1–14`) impression ratio. | Filled with `1.0` (neutral) when early 14d baseline is missing/zero. | Strictly prior: both windows `<= 2026-03-21`. |
| `ga4_sessions_21d_avg` | Float (continuous) | Mean daily GA4 sessions when `ga4_data_available IS TRUE`. | Filled with `0.0` when GA4 is unavailable (`is_ga4_connected=0`). | Strictly prior: `<= 2026-03-21`. |
| `ga4_engagement_pct` | Float (`[0, 100]`) | Engaged session rate (`100 * engaged / sessions`, ×100 scale). | Filled with `0.0` when GA4 is unavailable or sessions = 0. | Strictly prior: `<= 2026-03-21`. |
| `is_ga4_connected` | Binary (`0/1`) | Flags whether `ga4_data_available IS TRUE` occurred in `[Mar 1, Mar 21]`. | Deterministic integer (`0` or `1`), no missing values. | Strictly prior: `<= 2026-03-21`. |
| `has_full_21d_history` | Binary (`0/1`) | Flags whether the URL had all 21 days of GSC telemetry before cutoff. | Deterministic integer (`0` or `1`), no missing values. | Strictly prior: count over `<= 2026-03-21`. |
| `is_position_missing` | Binary (`0/1`) | Flags URLs where `gsc_avg_position` was `0` ("no data") throughout pre-window. | Deterministic integer (`0` or `1`), no missing values. | Strictly prior: `<= 2026-03-21`. |
| `tier_top_3` .. `tier_unranked` | Categorical (5 one-hot cols) | Discretized SERP rank tier (`top_3`, `page_1_rest`, `striking_distance`, `page_3_plus`, `unranked`). | Missing positions route deterministically to `tier_unranked`. | Strictly prior: derived from `position_21d_avg`. |

In [4]:
# Programmatic integrity audit of the constructed feature matrix X
audit_summary = pd.DataFrame({
    "feature": X.columns,
    "dtype": X.dtypes.astype(str).values,
    "missing_count": X.isna().sum().values,
    "inf_count": np.isinf(X.to_numpy()).sum(axis=0),
    "mean": X.mean().round(4).values,
    "std": X.std().round(4).values,
    "min": X.min().round(4).values,
    "max": X.max().round(4).values,
})

assert audit_summary["missing_count"].sum() == 0, "QUALITY ERROR: Residual NaNs found in X."
assert audit_summary["inf_count"].sum() == 0, "QUALITY ERROR: Infinite values found in X."

print("[Feature Matrix Integrity Audit — PASSED]")
print(audit_summary.to_string(index=False))

[Feature Matrix Integrity Audit — PASSED]
                   feature   dtype  missing_count  inf_count     mean      std     min        max
       impressions_21d_avg float64              0          0 116.5348 239.7472 10.0000 14369.0526
        impressions_21d_cv float64              0          0   0.4940   0.2860  0.0000     4.3503
            clicks_21d_sum float64              0          0   7.0982  25.9338  0.0000  3398.0000
          position_21d_avg float64              0          0  12.3519  12.1119  0.1625    97.7677
         best_position_21d float64              0          0   5.4225   8.1152  0.0003    87.6667
               ctr_21d_pct float64              0          0   0.2995   0.4458  0.0000    15.8730
impressions_momentum_ratio float64              0          0   1.1297   0.9724  0.0005    10.0000
      ga4_sessions_21d_avg float64              0          0   1.3390   3.5083  0.0000   125.2000
        ga4_engagement_pct float64              0          0   1.4370   7.61

## 3. The Leakage Hunt

We evaluate our clean feature matrix `X` under a strict **client-grouped holdout split (`GroupShuffleSplit` on `client_hash_id`)** using `PR-AUC` and `ROC-AUC`, and run three adversarial leakage stress tests:
1. **Attack 1 (Label-Derived Future Ratio):** Injects `(post_impressions_mean - raw_impressions_21d_avg) / raw_impressions_21d_avg`, which directly encodes the post-cutoff outcome.
2. **Attack 2 (Future Window Contamination):** Injects a leaky feature computed over the post-cutoff window (`report_date > 2026-03-21`).
3. **Attack 3 (Single-Feature Sentinel):** Evaluates every individual column in `X` independently on held-out clients to ensure no single feature achieves suspicious near-perfect discrimination (`PR-AUC >= 0.95` or `ROC-AUC >= 0.95`).

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

# 1. Client-Grouped Honest Split (no client appears in both train and validation)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, val_idx = next(gss.split(X, y, groups=groups_client))

X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

# Baseline Honest Model
honest_rf = RandomForestClassifier(n_estimators=60, max_depth=6, random_state=42, n_jobs=-1)
honest_rf.fit(X_train, y_train)
pred_honest = honest_rf.predict_proba(X_val)[:, 1]
prauc_honest = average_precision_score(y_val, pred_honest)
rocauc_honest = roc_auc_score(y_val, pred_honest)

# Attack 1: Direct Label-Derived Future Ratio
leaked_ratio = (
    (df_feat["post_impressions_mean"] - df_feat["raw_impressions_21d_avg"])
    / df_feat["raw_impressions_21d_avg"].replace(0, np.nan)
).fillna(0.0)

X_train_atk1 = X_train.copy()
X_val_atk1 = X_val.copy()
X_train_atk1["leaked_future_pct_change"] = leaked_ratio.iloc[train_idx].values
X_val_atk1["leaked_future_pct_change"] = leaked_ratio.iloc[val_idx].values

atk1_rf = RandomForestClassifier(n_estimators=60, max_depth=6, random_state=42, n_jobs=-1)
atk1_rf.fit(X_train_atk1, y_train)
pred_atk1 = atk1_rf.predict_proba(X_val_atk1)[:, 1]
prauc_atk1 = average_precision_score(y_val, pred_atk1)
rocauc_atk1 = roc_auc_score(y_val, pred_atk1)

# Attack 2: Future Window Contamination (post-cutoff mean impressions leaked as feature)
X_train_atk2 = X_train.copy()
X_val_atk2 = X_val.copy()
X_train_atk2["leaked_post_impressions_mean"] = df_feat["post_impressions_mean"].iloc[train_idx].values
X_val_atk2["leaked_post_impressions_mean"] = df_feat["post_impressions_mean"].iloc[val_idx].values

atk2_rf = RandomForestClassifier(n_estimators=60, max_depth=6, random_state=42, n_jobs=-1)
atk2_rf.fit(X_train_atk2, y_train)
pred_atk2 = atk2_rf.predict_proba(X_val_atk2)[:, 1]
prauc_atk2 = average_precision_score(y_val, pred_atk2)
rocauc_atk2 = roc_auc_score(y_val, pred_atk2)

# Attack 3: Single-Feature Leakage Sentinel across all clean features in X
single_feature_roc = {}
for col in FEATURE_COLS:
    stump = RandomForestClassifier(n_estimators=20, max_depth=3, random_state=42, n_jobs=-1)
    stump.fit(X_train[[col]], y_train)
    single_feature_roc[col] = roc_auc_score(y_val, stump.predict_proba(X_val[[col]])[:, 1])

max_single_feat = max(single_feature_roc, key=single_feature_roc.get)
max_single_roc = single_feature_roc[max_single_feat]

# Structural SQL & Feature Assertions (ignores SQL comments)
sql_no_comments = "\n".join(line.split("--")[0] for line in query_feature_vector.splitlines())
sql_body_without_target = sql_no_comments.split("post_impressions_mean")[0].upper()
assert "LEAD(" not in sql_body_without_target, "LEAKAGE DETECTED: LEAD() found inside feature definitions."
assert "2026-03-22" not in sql_body_without_target, "LEAKAGE DETECTED: Future date found inside feature definitions."
assert max_single_roc < 0.90, f"LEAKAGE SENTINEL ALERT: Single feature '{max_single_feat}' achieved suspicious ROC-AUC={max_single_roc:.4f}."
leakage_report = pd.DataFrame([
    {"experiment": "Honest Feature Vector (Grouped Client Val)", "pr_auc": round(prauc_honest, 4), "roc_auc": round(rocauc_honest, 4), "verdict": "CLEAN (Decision-time causal)"},
    {"experiment": "Attack 1: Label-Derived Future Ratio", "pr_auc": round(prauc_atk1, 4), "roc_auc": round(rocauc_atk1, 4), "verdict": "LEAKED (Artificially near 1.0)"},
    {"experiment": "Attack 2: Post-Cutoff Window Contamination", "pr_auc": round(prauc_atk2, 4), "roc_auc": round(rocauc_atk2, 4), "verdict": "LEAKED (Future window leak)"},
])

print(f"[Leakage Hunt Summary — Positive Base Rate: {y_val.mean():.4f}]")
print(leakage_report.to_string(index=False))
print(f"\n[Single-Feature Sentinel] Highest standalone validation ROC-AUC: '{max_single_feat}' = {max_single_roc:.4f} (threshold < 0.90 — PASSED)")

[Leakage Hunt Summary — Positive Base Rate: 0.3323]
                                experiment  pr_auc  roc_auc                        verdict
Honest Feature Vector (Grouped Client Val)  0.7060   0.8076   CLEAN (Decision-time causal)
      Attack 1: Label-Derived Future Ratio  1.0000   1.0000 LEAKED (Artificially near 1.0)
Attack 2: Post-Cutoff Window Contamination  0.8767   0.9138    LEAKED (Future window leak)

[Single-Feature Sentinel] Highest standalone validation ROC-AUC: 'impressions_momentum_ratio' = 0.7798 (threshold < 0.90 — PASSED)


## 4. What I Excluded and Why

* **`raw_client_domain` / client names / page URLs:** Excluded to protect FlyRank client confidentiality and prevent domain-level baseline memorization.
* **`query_text_unhashed` / raw search queries:** Excluded due to privacy rules on client query logs and extreme high-cardinality sparsity.
* **`client_hash_id`:** Used strictly as the `GroupShuffleSplit` key and excluded from `X` so the model generalizes to unseen clients.
* **`content_hash_id`:** Used strictly as the aggregation grain key and excluded from `X` to prevent per-URL memorization.
* **`report_date`:** Used only to enforce the `2026-03-21` temporal cutoff and excluded as a raw feature.
* **`gsc_data_available` / `ga4_data_available`:** Used strictly as boolean pre-filters (`IS TRUE`) so pre-onboarding zero-filled rows are excluded rather than misread as zero traffic.
* **`post_impressions_mean` & `is_future_decay` (`2026-03-22` to `2026-03-31`):** Ground-truth outcome window signals occurring strictly after the decision cutoff.
* **Raw `gsc_avg_position == 0`:** Encodes *"no ranking data"* rather than rank #0; converted via `NULLIF(gsc_avg_position, 0)` prior to aggregation.
* **Post-hoc refresh / product action flags (e.g., `refresh_triggered`, `intervention_status`):** Excluded because editorial refresh actions are logged *after* observed traffic drops, creating reverse-causality target leakage.

In [6]:
# Explicit Exclusion Registry & Automated Privacy/Leakage Guardrail
EXCLUDED_FIELDS = {
    "raw_client_domain": "Privacy & memorization risk: exposes client identity and leaks domain-level baseline traffic.",
    "query_text_unhashed": "Privacy & sparsity risk: raw search queries violate FlyRank confidentiality and add extreme cardinality.",
    "client_hash_id": "Grouping key only: excluded from X so models generalize to unseen clients rather than memorizing IDs.",
    "content_hash_id": "Grain key only: excluded from X to prevent URL-level memorization.",
    "report_date": "Cutoff ordering key only: raw date excluded to avoid spurious monotonic calendar drift.",
    "gsc_data_available": "Contract pre-filter only (WHERE gsc_data_available IS TRUE); constant after filtering.",
    "post_impressions_mean": "Post-cutoff outcome window metric (> 2026-03-21); occurs strictly after prediction timestamp.",
    "is_future_decay": "Ground-truth target label derived from post-cutoff window.",
    "post_hoc_refresh_flags": "Operational action flags recorded after traffic drops induce reverse-causality leakage.",
}

# Guardrail 1: Zero overlap between excluded fields and active model features
overlap = set(EXCLUDED_FIELDS.keys()) & set(FEATURE_COLS)
assert len(overlap) == 0, f"CONTRACT VIOLATION: Excluded fields found in FEATURE_COLS: {overlap}"

# Guardrail 2: Scan feature names for forbidden substrings
forbidden_tokens = ("target", "lead", "future", "post_", "following", "domain", "url", "query", "client_hash_id", "content_hash_id")
flagged_names = [
    col for col in FEATURE_COLS
    if any(tok in col.lower() for tok in forbidden_tokens)
]
assert len(flagged_names) == 0, f"PRIVACY/LEAKAGE VIOLATION: Suspicious feature names detected: {flagged_names}"

print("[Exclusion & Privacy Guardrails — PASSED]")
print(f"Verified {len(EXCLUDED_FIELDS)} excluded fields against {len(FEATURE_COLS)} active features with zero overlap.")

[Exclusion & Privacy Guardrails — PASSED]
Verified 9 excluded fields against 17 active features with zero overlap.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.